In [1]:
import os
import zipfile
import cv2
import numpy as np
import pandas as pd
from skimage.feature import graycomatrix, graycoprops
from skimage.metrics import peak_signal_noise_ratio as psnr

# 1. Path to your uploaded zip
zip_path = '/content/dataset_cv.zip'
extract_path = '/content/dataset_extracted'

# 2. Unzip the file
if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("Step 1: Extraction Complete.")
else:
    print(" Error: Zip file not found at the path provided!")

# 3. Create a folder to save processed images for your GitHub
os.makedirs('/content/processed_samples', exist_ok=True)

✅ Step 1: Extraction Complete.


In [4]:
def process_cv_pipeline(img_path, filename):
    # Load image
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None: return None

    # Assignment 1: Radiometric Cleaning
    # Identifying Bit-depth
    bit_depth = img.dtype
    # Filtering (Median for sensor noise)
    denoised = cv2.medianBlur(img, 5)
    # Gaussian for Anti-Aliasing
    clean_img = cv2.GaussianBlur(denoised, (5, 5), 0)
    # Calculate PSNR
    noise_score = psnr(img, clean_img, data_range=img.max() - img.min())

    #  Assignment 2: Geometry & Chain Code
    # Segmentation & Morphological Cleaning
    _, mask = cv2.threshold(clean_img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((5,5), np.uint8))

    # Find Contours
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not contours: return None
    cnt = max(contours, key=cv2.contourArea)

    # Convex Hull (Computational Geometry)
    hull = cv2.convexHull(cnt)

    # 8-Directional Chain Code (Sample of first 10 directions)
    chain_code = []
    for i in range(min(len(cnt)-1, 20)): # Taking a small sample for the CSV
        p1, p2 = cnt[i][0], cnt[i+1][0]
        dx, dy = np.sign(p2[0]-p1[0]), np.sign(p2[1]-p1[1])
        lookup = {(1,0):0, (1,1):1, (0,1):2, (-1,1):3, (-1,0):4, (-1,-1):5, (0,-1):6, (1,-1):7}
        chain_code.append(lookup.get((dx, dy), 0))

    # Assignment 3: Statistical Features (GLCM)
    glcm = graycomatrix(clean_img, [1], [0], 256, symmetric=True, normed=True)

    # Save one processed image as a sample for your repo
    cv2.imwrite(f'/content/processed_samples/cleaned_{filename}', clean_img)

    return {
        "filename": filename,
        "bit_depth": str(bit_depth),
        "psnr": noise_score,
        "area": cv2.contourArea(cnt),
        "energy": graycoprops(glcm, 'energy')[0, 0],
        "contrast": graycoprops(glcm, 'contrast')[0, 0],
        "chain_code": str(chain_code)
    }

In [5]:
results = []
for root, dirs, files in os.walk(extract_path):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg', '.tif')):
            file_path = os.path.join(root, file)
            data = process_cv_pipeline(file_path, file)
            if data:
                results.append(data)

# Create and save CSV
df = pd.DataFrame(results)
df.to_csv('Feature_Vector_Report.csv', index=False)
print(f"Processed {len(results)} images. Download 'Feature_Vector_Report.csv' now!")

✅ Processed 26 images. Download 'Feature_Vector_Report.csv' now!
